In [1]:
import sqlite3
import pandas as pd

In [2]:
# Update if any changes to location of data
AIRBNB_CSV  = "sa_aug_cleaned_chch_airbnb.csv"
TENANCY_CSV = "../data/tenancy_services_chch_only.csv"

DB_PATH     = "../rentals.db"

# Opens a connection
con = sqlite3.connect(DB_PATH)

In [3]:
# Defines the AirBnb table with types for each column
con.execute("DROP TABLE IF EXISTS airbnb_listing_month;")
con.execute("""
CREATE TABLE airbnb_listing_month (
    id                              INTEGER NOT NULL,
    name                            TEXT,
    host_id                         REAL,
    host_name                       TEXT,
    neighbourhood_group             TEXT,
    neighbourhood                   TEXT,
    latitude                        REAL,
    longitude                       REAL,
    room_type                       TEXT,
    price                           REAL,
    minimum_nights                  REAL,
    number_of_reviews               INTEGER,
    last_review                     TEXT,
    reviews_per_month               REAL,
    calculated_host_listings_count  REAL,
    availability_365                INTEGER,
    number_of_reviews_ltm           INTEGER,
    license                         TEXT,
    year_month                      TEXT NOT NULL,
    quarter                         TEXT NOT NULL,
    SA22019_V1_00                   INTEGER,
    PRIMARY KEY (id, year_month)
);
""");

In [4]:
# Reads file then removes pandas index column
df = pd.read_csv(AIRBNB_CSV)
df = df.drop(columns="Unnamed: 0")

# loads rows into table then puts prepared dataset into database
df.to_sql("airbnb_listing_month", con, if_exists="append", index=False)
con.commit()

In [5]:
# checks table has all 28,795 rows, that join columns are stored as integer and text, that there are 3 quarters, if so then load is successful
print(pd.read_sql_query("SELECT COUNT(*) AS n_rows FROM airbnb_listing_month;", con))
print(pd.read_sql_query("SELECT typeof(SA22019_V1_00) AS sa_type, typeof(quarter) AS q_type FROM airbnb_listing_month LIMIT 1;", con))
print(pd.read_sql_query("SELECT COUNT(DISTINCT quarter) AS n_quarters FROM airbnb_listing_month;", con))

   n_rows
0   28795
   sa_type q_type
0  integer   text
   n_quarters
0           3


In [6]:
# creates an empty tenancy table with fixed types, read for next dataset
con.execute("DROP TABLE IF EXISTS tenancy_bond;")
con.execute("""
CREATE TABLE tenancy_bond (
    timeframe               TEXT    NOT NULL,
    location_id             INTEGER NOT NULL,
    dwelling_type           TEXT    NOT NULL,
    number_of_beds          TEXT,
    total_bonds             INTEGER,
    active_bonds            INTEGER,
    closed_bonds            INTEGER,
    median_rent             REAL,
    geometric_mean_rent     REAL,
    upper_quartile_rent     REAL,
    lower_quartile_rent     REAL,
    log_std_dev_weekly_rent REAL
);
  """);

In [7]:
# reads tenancy file then converts column names to lowercase with underscores so that it can be used in SQL
# data is then loaded as the second dataset in the database
ts = pd.read_csv(TENANCY_CSV)
ts.columns = ts.columns.str.lower().str.replace(" ", "_")
ts.to_sql("tenancy_bond", con, if_exists="append", index=False)
con.commit()

In [8]:
# check that all 21,828 rows are loaded and have correct types
print(pd.read_sql_query("SELECT COUNT(*) AS n_rows FROM tenancy_bond;", con))
print(pd.read_sql_query("SELECT typeof(location_id) AS id_type,typeof(timeframe) AS tf_type FROM tenancy_bond LIMIT 1;", con))

   n_rows
0   21828
   id_type tf_type
0  integer    text


In [9]:
# SQL query that groups monthly airbnb rows into 1 row per listing per quarter,
# then filtered tenancy to 1 "all dwellings, all bedrooms" rent per area per quarter with correct quarter format - without this it would create copies of listings, all with different rents.
# then left joins them on area code and quarter so every listing-quarter has its area's median rent without any rows being lost or duplicated.
join_sql = """
WITH airbnb_ranked AS (
      SELECT *,
          ROW_NUMBER() OVER (PARTITION BY id, quarter ORDER BY year_month) AS month_order
      FROM airbnb_listing_month
),
airbnb_quarterly AS (
      SELECT
          id,
          quarter,
          MIN(year_month)         AS year_month,
          AVG(price)              AS price,
          AVG(minimum_nights)     AS minimum_nights,
          MAX(number_of_reviews)  AS number_of_reviews,
          MAX(CASE WHEN month_order = 1 THEN SA22019_V1_00 END) AS SA22019_V1_00
      FROM airbnb_ranked
      GROUP BY id, quarter
),
tenancy_quarterly AS (
      SELECT
          location_id,
          strftime('%Y', timeframe) || 'Q' ||
              ((CAST(strftime('%m', timeframe) AS INTEGER) + 2) / 3) AS quarter,
          total_bonds,
          active_bonds,
          median_rent,
          lower_quartile_rent,
          upper_quartile_rent
      FROM tenancy_bond
      WHERE dwelling_type = 'ALL' AND number_of_beds = 'ALL'
)
SELECT
      a.id, a.quarter, a.year_month, a.SA22019_V1_00,
      a.price, a.minimum_nights, a.number_of_reviews,
      t.median_rent, t.lower_quartile_rent, t.upper_quartile_rent,
      t.total_bonds, t.active_bonds
FROM airbnb_quarterly AS a
LEFT JOIN tenancy_quarterly AS t
      ON  a.SA22019_V1_00 = t.location_id
      AND a.quarter       = t.quarter;
"""
joined = pd.read_sql_query(join_sql, con)

In [10]:
# Checks the joined result has all the expected listings-quarters and shows how many have got a median rent and how many did not.
# Confirms there are no listing-quarter duplicates and proving the join did not lose any rows or duplicate any.
print("rows:", len(joined))
print("matched:", joined["median_rent"].notna().sum())
print("unmatched:", joined["median_rent"].isna().sum(), "| unmatched SA2s:", joined.loc[joined["median_rent"].isna(), "SA22019_V1_00"].nunique())
print("duplicate listing-quarters:", joined.duplicated(["id", "quarter"]).sum())

rows: 10355
matched: 8861
unmatched: 1494 | unmatched SA2s: 46
duplicate listing-quarters: 0
